In [1]:
from utils.data import ProteinDataset, ProteinPairDataset, pair_collate_fn
import torch as pt


data = pt.load(f'./data/engineered_data.pt')
datalib = ProteinDataset(data)
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)

/tmp/ipykernel_3213602/2873144890.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = pt.load(f'./data/engineered_data.pt')


In [5]:
import pandas as pd


df = pd.read_csv('./data/tmalign.out', names=['name1', 'name2', 'tmscore', 'seqid'], delimiter='\t')
df['name1'] = df['name1'].str.split('/').str[-1].str.removesuffix('.pdb')
df['name2'] = df['name2'].str.split('/').str[-1].str.removesuffix('.pdb')
df['idx1'] = df['name1'].map(pdb2idx)
df['idx2'] = df['name2'].map(pdb2idx)
df_with_nan = df[df.isna().any(axis=1)]

In [6]:
print(df_with_nan)

          name1   name2  tmscore  seqid     idx1     idx2
1761     5A1U-F  5A1U-H   0.7622  0.144  32403.0      NaN
1779     5A1U-H  5A1U-F   0.7807  0.148      NaN  32403.0
1780     5A1U-H  4AFI-A   0.6776  0.059      NaN  26346.0
1781     5A1U-H  7B6D-E   0.6433  0.067      NaN  48318.0
1782     5A1U-H  7B6D-H   0.6513  0.104      NaN  48372.0
...         ...     ...      ...    ...      ...      ...
1206756  4ZRY-B  5GAR-I   0.6154  0.058  55698.0      NaN
1207009  6ZR2-E  5LUF-E   0.7186  0.019  55714.0      NaN
1209712  5ZUO-A  6N88-C   0.6462  0.137  54739.0      NaN
1210919  6ZVH-y  6N88-C   0.6703  0.125  93672.0      NaN
1212145  1ZX2-A  7Q1V-D   0.6225  0.100  34391.0      NaN

[2354 rows x 6 columns]


In [2]:
import torch as pt
import torch.nn as nn
import torch_geometric.nn as gnn


class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=256, hidden_channels:int=256, num_layers:int=3, ):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=21, embedding_dim=embed_dim, padding_idx=0)
        # node_attr占一维
        self.gcn = gnn.GCN(in_channels=embed_dim+1, hidden_channels=hidden_channels, 
                           num_layers=num_layers, out_channels=embed_dim)
        self.shared = nn.Sequential(nn.Linear(4*embed_dim, embed_dim), nn.ReLU(),)
        self.tm_head = nn.Linear(embed_dim, 1)
        self.seq_head = nn.Linear(embed_dim, 1)

    def _embed_(self, seq_mask):
        seq, mask = seq_mask
        embedding = self.emb(seq) # [batch_size, seq_len, emb_dim]
        embedding = embedding * mask.unsqueeze(-1) # mask: [batch_size, seq_len, 1]
        return embedding

    def embed(self, seq_mask):
        self.eval()
        with pt.no_grad():
            embedding = self._embed_(seq_mask)
        return embedding

    def forward(self, data):
        seq_mask_graph_i, seq_mask_graph_j = data
        seq_i, mask_i, graph_i = seq_mask_graph_i
        seq_j, mask_j, graph_j = seq_mask_graph_j
        x_i, edge_index_i, edge_attr_i, batch_i, node2seq_i = graph_i.x, graph_i.edge_index, graph_i.edge_attr, graph_i.batch, graph_i.node2seq
        x_j, edge_index_j, edge_attr_j, batch_j, node2seq_j = graph_j.x, graph_j.edge_index, graph_j.edge_attr, graph_j.batch, graph_j.node2seq
        # [batch_size, seq_len, emb_dim]
        emb_i = self._embed_((seq_i, mask_i))
        emb_j = self._embed_((seq_j, mask_j))
        B_i, L_i, D_i = emb_i.size()
        B_j, L_j, D_j = emb_j.size()
        # [total_num_nodes, emb_dim]
        emb_flat_i = emb_i.view(-1, D_i)
        emb_flat_j = emb_j.view(-1, D_j)
        flat_idx_i = batch_i * L_i + node2seq_i
        flat_idx_j = batch_j * L_j + node2seq_j
        node_emb_i = emb_flat_i[flat_idx_i]
        node_emb_j = emb_flat_j[flat_idx_j]
        x_i = pt.cat([node_emb_i, x_i], dim=-1)
        x_j = pt.cat([node_emb_j, x_j], dim=-1)
        x_i = self.gcn(x_i, edge_index_i, edge_attr = edge_attr_i, batch = batch_i)
        x_j = self.gcn(x_j, edge_index_j, edge_attr = edge_attr_j, batch = batch_j)
        # [batch_size, emb_dim]
        x_i = gnn.global_mean_pool(x_i, batch_i)
        x_j = gnn.global_mean_pool(x_j, batch_j)
        feature = pt.cat([x_i, x_j, pt.abs(x_i - x_j), x_i * x_j], dim=-1)
        shared_feat = self.shared(feature)
        tm_score = pt.sigmoid(self.tm_head(shared_feat)).squeeze(-1)
        seq_score = self.seq_head(shared_feat).squeeze(-1)
        return tm_score, seq_score

In [3]:
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import DataLoader



pair_dataset = ProteinPairDataset(datalib, './data/tmalign.out', pdb2idx)
loader = DataLoader(pair_dataset, batch_size=256, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)
gpu = 6
data_map = np.arange(len(pair_dataset), dtype=np.int64)
train_map, test_map = train_test_split(data_map, test_size=10240, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=train_map)
test_set = ProteinPairDataset(pair_dataset, mapping=test_map)
batch_size = 1024
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=pair_collate_fn, drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)

In [4]:
import logging
from tqdm import tqdm


lamda = 0.1
prot_model = ProteinGCN().to(gpu)
criterion = nn.SmoothL1Loss()
learning_rate = 1e-3
optimizer = pt.optim.AdamW(prot_model.parameters(), lr=learning_rate)
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = []
    prot_model.train()
    for i, batch in enumerate(tqdm(train_loader, unit='batch')):
        data_i, data_j, label = batch
        seqs_i, masks_i, graphs_i = data_i
        seqs_j, masks_j, graphs_j = data_j
        seqs_i, masks_i, graphs_i = seqs_i.to(gpu), masks_i.to(gpu), graphs_i.to(gpu)
        seqs_j, masks_j, graphs_j = seqs_j.to(gpu), masks_j.to(gpu), graphs_j.to(gpu)
        label = label.to(gpu)
        output = prot_model(((seqs_i, masks_i, graphs_i, ), (seqs_j, masks_j, graphs_j, )))
        tm_score, seq_score = output
        tm_loss = criterion(tm_score, label[:, 0])
        seq_loss = criterion(seq_score, label[:, 1])
        loss = tm_loss + lamda * seq_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())
        if (i+1) % 100 == 0:
            l = pt.tensor(train_loss).mean()
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {l:.4f}')
            train_loss = []

  9%|▊         | 100/1176 [00:29<04:41,  3.82batch/s]

Epoch [1/10], Train Loss: 0.0038


 10%|▉         | 114/1176 [00:34<05:16,  3.35batch/s]


KeyboardInterrupt: 

In [ ]:
pt.cuda.empty_cache()